# **step:-1 Import Libraries**

In [1]:
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori,association_rules,fpgrowth

import warnings

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    module="jupyter_client"
)

# **Step 2: Load Dataset**

In [2]:
df=pd.read_csv('blinkit_market_basket_dataset.csv')
df.head()

,Transaction_ID,Customer_ID,Date,Product_ID,Product_Name,Quantity,Unit_Price,Category
0,T0000001,C05875,2025-09-21,P0027,Cookies,3,32.37,Snacks
1,T0000001,C05875,2025-09-21,P0024,Chocolate Bar,3,26.34,Snacks
2,T0000001,C05875,2025-09-21,P0037,Juice,2,101.46,Beverages
3,T0000001,C05875,2025-09-21,P0111,Frozen Corn,3,102.42,Frozen Foods
4,T0000001,C05875,2025-09-21,P0036,Soft Drink,2,50.63,Beverages


# **Step 3: Understand Dataset**

In [3]:
print(f"Total no of rows: {df.shape[0]}")
print(f"Total no of columns: {df.shape[1]}")

Total no of rows: 500000
Total no of columns: 8


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Transaction_ID  500000 non-null  object 
 1   Customer_ID     500000 non-null  object 
 2   Date            500000 non-null  object 
 3   Product_ID      500000 non-null  object 
 4   Product_Name    500000 non-null  object 
 5   Quantity        500000 non-null  int64  
 6   Unit_Price      500000 non-null  float64
 7   Category        500000 non-null  object 
dtypes: float64(1), int64(1), object(6)
memory usage: 30.5+ MB


In [5]:
df.describe()

,Quantity,Unit_Price
count,500000.000000,500000.000000
mean,2.301760,106.556191
std,1.131153,82.983153
min,1.000000,12.160000
25%,1.000000,42.450000
50%,2.000000,79.310000
75%,3.000000,152.800000
max,5.000000,470.290000


In [6]:
total_null_value=df.isnull().sum().sum()
print(f"Total Null value: {total_null_value}")

Total Null value: 0


In [7]:
total_duplicate_value=df.duplicated().sum()
print(f"Total Duplicated value: {total_duplicate_value}")

Total Duplicated value: 0


In [8]:
total_transaction=df['Transaction_ID'].nunique()
print(f"Total no of unique Transaction: {total_transaction}")

Total no of unique Transaction: 123912


In [9]:
l = []

for i in df['Product_Name']:
    l.append(i)

s = set(l)

print(len(s))

total_product_name = pd.DataFrame({
    "ID": range(1, len(s) + 1),
    "Product_Name": list(s)
})

97


# **Step 4: Data Cleaning**

The provided CSV dataset is already clean, with no significant missing or inconsistent values. Hence, no additional data cleaning is required, and we can proceed directly to the next step of the analysis.

# **Step 5: Removing Unnecessary columns**

In [10]:
df=df[['Transaction_ID','Product_Name']]
df.head()

,Transaction_ID,Product_Name
0,T0000001,Cookies
1,T0000001,Chocolate Bar
2,T0000001,Juice
3,T0000001,Frozen Corn
4,T0000001,Soft Drink


# **Step 6: Create Transaction Basket**

In [11]:
unique_products_per_transaction=df.groupby('Transaction_ID')['Product_Name'].nunique()
unique_products_per_transaction.head()

,Product_Name
Transaction_ID,
T0000001,5
T0000002,10
T0000003,5
T0000004,6
T0000005,7


In [12]:
basket=df.groupby('Transaction_ID')['Product_Name'].apply(list)
basket

,Product_Name
Transaction_ID,
T0000001,"[Cookies, Chocolate Bar, Juice, Frozen Corn, S..."
T0000002,"[Sugar 1kg, Milk 500ml, Bread, Tea, White Brea..."
T0000003,"[Milk 1L, Toor Dal 1kg, Tomatoes 1kg, Curd 400..."
T0000004,"[Salt 1kg, Onions 1kg, Turmeric Powder, Rice 1..."
T0000005,"[Body Wash, Garbage Bags, Floor Cleaner, Bathi..."
...,...
T0123908,"[Wheat Flour 5kg, Salt 1kg, Turmeric Powder, G..."
T0123909,"[Cumin Seeds, Wheat Flour 5kg, Onions 1kg, Tur..."
T0123910,"[Toothbrush, Toothpaste]"


In [13]:
te=TransactionEncoder()
basket_array=te.fit(basket).transform(basket)
basket_array

array([[False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False,  True],
       [False, False, False, ..., False, False, False],
       ...,
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]])

# **Step 7: Transaction Encoding**

In [14]:
basket_encode=pd.DataFrame(
    basket_array,
    columns=te.columns_,
    index=basket.index,
    dtype=bool
    )
basket_encode.head()

,Air Freshener,Apples 1kg,Baby Diapers,Baby Food,Baby Lotion,Baby Powder,Baby Shampoo,Baby Soap,Baby Wipes,Bananas 1kg,...,Sugar 1kg,Tea,Toilet Cleaner,Tomatoes 1kg,Toor Dal 1kg,Toothbrush,Toothpaste,Turmeric Powder,Wheat Flour 5kg,White Bread
Transaction_ID,,,,,,,,,,,,,,,,,,,,,
T0000001,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
T0000002,False,False,False,False,False,False,False,False,False,False,...,True,True,False,False,False,False,False,False,False,True
T0000003,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,True,False,False,False,False,False
T0000004,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,True,False,False
T0000005,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


# **Step 7: Apply Apriori Algorithm and Generate Association Rules**

In [15]:
frequent_itemsets_apriori=apriori(
    basket_encode,
    min_support=0.01,
    use_colnames=True,
    max_len=3,
    low_memory=True
)
frequent_itemsets_apriori.head()

,support,itemsets
0,0.016980,(Air Freshener)
1,0.030465,(Apples 1kg)
2,0.078007,(Baby Diapers)
3,0.028988,(Baby Food)
4,0.028157,(Baby Lotion)


In [16]:
rules_apriori=association_rules(
    frequent_itemsets_apriori,
    min_threshold=0.3,
    metric="confidence",
)
rules_apriori

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Apples 1kg),(Milk 1L),0.030465,0.152076,0.010830,0.355497,2.337630,1.0,0.006197,1.315624,0.590197,0.063073,0.239905,0.213356
1,(Baby Food),(Baby Diapers),0.028988,0.078007,0.016439,0.567094,7.269780,1.0,0.014178,2.129774,0.888191,0.181535,0.530467,0.388916
2,(Baby Lotion),(Baby Diapers),0.028157,0.078007,0.016221,0.576096,7.385190,1.0,0.014025,2.175006,0.889644,0.180350,0.540231,0.392021
3,(Baby Powder),(Baby Diapers),0.021709,0.078007,0.012339,0.568401,7.286547,1.0,0.010646,2.136228,0.881906,0.141221,0.531885,0.363292
4,(Baby Shampoo),(Baby Diapers),0.021725,0.078007,0.012033,0.553863,7.100177,1.0,0.010338,2.066616,0.878238,0.137204,0.516117,0.354058
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
280,"(Rice 5kg, Onions 1kg)",(Wheat Flour 5kg),0.028238,0.080210,0.010096,0.357531,4.457425,1.0,0.007831,1.431648,0.798194,0.102650,0.301504,0.241699
281,"(Rice 5kg, Wheat Flour 5kg)",(Onions 1kg),0.026664,0.103404,0.010096,0.378632,3.661675,1.0,0.007339,1.442939,0.746814,0.084152,0.306970,0.238134
282,"(Onions 1kg, Wheat Flour 5kg)",(Rice 5kg),0.028084,0.080501,0.010096,0.359483,4.465587,1.0,0.007835,1.435557,0.798490,0.102507,0.303407,0.242448
283,"(Onions 1kg, Wheat Flour 5kg)",(Tomatoes 1kg),0.028084,0.103606,0.011468,0.408333,3.941221,1.0,0.008558,1.515032,0.767836,0.095388,0.339948,0.259510


In [17]:
rules_apriori = rules_apriori[
    [
        'antecedents',
        'consequents',
        'support',
        'confidence',
        'lift'
    ]
]
rules_apriori

,antecedents,consequents,support,confidence,lift
0,(Apples 1kg),(Milk 1L),0.010830,0.355497,2.337630
1,(Baby Food),(Baby Diapers),0.016439,0.567094,7.269780
2,(Baby Lotion),(Baby Diapers),0.016221,0.576096,7.385190
3,(Baby Powder),(Baby Diapers),0.012339,0.568401,7.286547
4,(Baby Shampoo),(Baby Diapers),0.012033,0.553863,7.100177
...,...,...,...,...,...
280,"(Rice 5kg, Onions 1kg)",(Wheat Flour 5kg),0.010096,0.357531,4.457425
281,"(Rice 5kg, Wheat Flour 5kg)",(Onions 1kg),0.010096,0.378632,3.661675
282,"(Onions 1kg, Wheat Flour 5kg)",(Rice 5kg),0.010096,0.359483,4.465587
283,"(Onions 1kg, Wheat Flour 5kg)",(Tomatoes 1kg),0.011468,0.408333,3.941221


# **Step 8: Apply FP-Growth Algorithm and Generate Association Rules**

In [18]:
frequent_itemsets_fpgrowth=fpgrowth(
    basket_encode,
    min_support=0.01,
    use_colnames=True,
    max_len=3,
    verbose=1
)
frequent_itemsets_fpgrowth.head()

88 itemset(s) from tree conditioned on items ()
2 itemset(s) from tree conditioned on items (Soft Drink)
1 itemset(s) from tree conditioned on items (Soft Drink, Namkeen)
0 itemset(s) from tree conditioned on items (Soft Drink, Potato Chips)
3 itemset(s) from tree conditioned on items (Chocolate Bar)
1 itemset(s) from tree conditioned on items (Chocolate Bar, Soft Drink)
0 itemset(s) from tree conditioned on items (Chocolate Bar, Potato Chips)
0 itemset(s) from tree conditioned on items (Chocolate Bar, Namkeen)
4 itemset(s) from tree conditioned on items (Juice)
0 itemset(s) from tree conditioned on items (Juice, Soft Drink)
0 itemset(s) from tree conditioned on items (Juice, Chocolate Bar)
0 itemset(s) from tree conditioned on items (Juice, Namkeen)
0 itemset(s) from tree conditioned on items (Juice, Potato Chips)
4 itemset(s) from tree conditioned on items (Cookies)
0 itemset(s) from tree conditioned on items (Cookies, Soft Drink)
0 itemset(s) from tree conditioned on items (Cookies,

,support,itemsets
0,0.074117,(Soft Drink)
1,0.058009,(Chocolate Bar)
2,0.039125,(Juice)
3,0.035808,(Cookies)
4,0.015664,(Frozen Corn)


In [19]:
rules_fpgrowth=association_rules(
    frequent_itemsets_fpgrowth,
    min_threshold=0.3,
    metric="confidence",
)
rules_fpgrowth

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Namkeen),(Soft Drink),0.079895,0.074117,0.024429,0.305758,4.125330,1.0,0.018507,1.333659,0.823379,0.188516,0.250183,0.317676
1,(Soft Drink),(Namkeen),0.074117,0.079895,0.024429,0.329595,4.125330,1.0,0.018507,1.372461,0.818241,0.188516,0.271382,0.317676
2,(Potato Chips),(Soft Drink),0.092816,0.074117,0.035711,0.384749,5.191097,1.0,0.028832,1.504887,0.889965,0.272140,0.335498,0.433283
3,(Soft Drink),(Potato Chips),0.074117,0.092816,0.035711,0.481816,5.191097,1.0,0.028832,1.750700,0.871992,0.272140,0.428800,0.433283
4,"(Namkeen, Potato Chips)",(Soft Drink),0.030836,0.074117,0.012469,0.404344,5.455480,1.0,0.010183,1.554393,0.842683,0.134817,0.356662,0.286286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
280,(Biscuits),(Tea),0.048204,0.051997,0.024211,0.502260,9.659485,1.0,0.021704,1.904616,0.941877,0.318607,0.474960,0.483941
281,(Biscuits),(Namkeen),0.048204,0.079895,0.017182,0.356437,4.461299,1.0,0.013330,1.429705,0.815143,0.154904,0.300555,0.285744
282,(Biscuits),(Milk 500ml),0.048204,0.061963,0.015067,0.312573,5.044488,1.0,0.012080,1.364562,0.842369,0.158435,0.267164,0.277868
283,(Nachos),(Potato Chips),0.026987,0.092816,0.013663,0.506280,5.454670,1.0,0.011158,1.837446,0.839322,0.128726,0.455766,0.326742


In [20]:
rules_fpgrowth = rules_fpgrowth[
    [
        'antecedents',
        'consequents',
        'support',
        'confidence',
        'lift'
    ]
]
rules_fpgrowth

,antecedents,consequents,support,confidence,lift
0,(Namkeen),(Soft Drink),0.024429,0.305758,4.125330
1,(Soft Drink),(Namkeen),0.024429,0.329595,4.125330
2,(Potato Chips),(Soft Drink),0.035711,0.384749,5.191097
3,(Soft Drink),(Potato Chips),0.035711,0.481816,5.191097
4,"(Namkeen, Potato Chips)",(Soft Drink),0.012469,0.404344,5.455480
...,...,...,...,...,...
280,(Biscuits),(Tea),0.024211,0.502260,9.659485
281,(Biscuits),(Namkeen),0.017182,0.356437,4.461299
282,(Biscuits),(Milk 500ml),0.015067,0.312573,5.044488
283,(Nachos),(Potato Chips),0.013663,0.506280,5.454670


# **Step 9: Top 3 Recommendation from Apriori Algo**

In [21]:
def top3_recommendation_apriori(product, rules):

    recommendations = rules[
        rules['antecedents'].apply(
            lambda x: product in x
        )
    ]

    recommendations = recommendations.sort_values(
        by=['lift', 'confidence'],
        ascending=False
    )

    if recommendations.empty:
        return []

    la = []

    for i in range(len(recommendations)):

        recommended_product = ", ".join(
            recommendations.iloc[i]['consequents']
        )

        if recommended_product not in la:
            la.append(recommended_product)

        if len(la) == 3:
            break

    return la

# **Step 10: Top 3 Recommendation from fpgrowth Algo**

In [22]:
def top3_recommendation_fpgrowth(product, rules):

    recommendations = rules[
        rules['antecedents'].apply(
            lambda x: product in x
        )
    ]

    recommendations = recommendations.sort_values(
        by=['lift', 'confidence'],
        ascending=False
    )

    if recommendations.empty:
        return []

    lf = []

    for i in range(len(recommendations)):

        recommended_product = ", ".join(
            recommendations.iloc[i]['consequents']
        )

        if recommended_product not in lf:
            lf.append(recommended_product)

        if len(lf) == 3:
            break

    return lf

# **Step 11: Final Product Recommendations**

In [23]:
print("Total products are:")

pd.set_option('display.max_rows', None)
display(total_product_name.style.hide(axis="index"))

Total products are:


ID,Product_Name
1,Cream Biscuits
2,Apples 1kg
3,Baby Powder
4,Deodorant
5,Jam
6,Energy Drink
7,Chilli Powder
8,Rusk
9,Frozen Nuggets
10,Pasta


In [27]:
product = input("Enter the search product: ")
top3_apriori = top3_recommendation_apriori(product, rules_apriori)
top3_fpgrowth = top3_recommendation_fpgrowth(product, rules_fpgrowth)
final_recommend = list(set(top3_apriori + top3_fpgrowth))

print("-" * 100)
print(f"Your Product: {product}")
print()
print("Recommendation Products:")
print()

if len(final_recommend) == 0:
    print("No Recommendation Found.")

for i in range(len(final_recommend)):
    print(f"{i + 1}. {final_recommend[i]}")

print("-" * 100)

Enter the search product: Garam Masala
----------------------------------------------------------------------------------------------------
Your Product: Garam Masala

Recommendation Products:

1. Rice 5kg
2. Cooking Oil 1L
3. Wheat Flour 5kg
----------------------------------------------------------------------------------------------------


# **Step 12: Conclusion**

# In this project, Market Basket Analysis was performed on the Blinkit grocery transaction dataset to identify customer purchasing patterns. The Apriori and FP-Growth algorithms were used to discover frequent itemsets and generate association rules using Support, Confidence, and Lift.

# Based on the generated association rules, a Product Recommendation System was developed to recommend products related to a selected product. Recommendations from both algorithms were combined, and duplicate recommendations were removed to generate the final recommendation list.

# Overall, this project demonstrates how Association Rule Mining can be effectively used to analyse customer purchasing behaviour and build a practical product recommendation system for grocery businesses.